# Stage A. Obtain the broad candidate employment-spell sample

The introduction of current notebook:

- This notebook creates a broad, downloadable employment-spell-level dataset for later local sample construction.
- It intentionally does not restrict industries or detailed occupations, remove internships, collapse records to the user-company level, or restrict the sample to the United States.

The Fabric-stage restrictions are limited to:

1. employment starts in `[2021-01-01, 2024-01-01)`;
2. usable position, user, company, position-number, country, and raw-or-translated-title information; and
3. a delivered O*NET code whose first two characters are `17` or `19`.

The output remains one row per retained source employment spell.

In [ ]:
"""
Task:
    Extract variables for broad candidate employment spells on Fabric.

Inputs:
(a) Fabric table `user_positions`.

Outputs:
(a) Files/WenzhiW/A01_BaselineUSBiopharma/StageA_BroadestCandidateEmpSpells/
(b) Files/WenzhiW/A01_BaselineUSBiopharma/StageA_BroadestCandidateEmpSpells.zip

Descriptions of outputs:
(1) Output (a) has one row per source employment spell meeting the Stage A restrictions.
(2) Output (b) packages every Parquet part of (a) for download.

Notes:
(1) Keep employment spells starting in [2021-01-01, 2024-01-01) and ONET major groups 17 or 19.
(2) Require non-missing `position_id`, `user_id`, `rcid`, `position_number`, `country`, and at least
    one job title (raw or translated).
(3) Apply no US, industry, internship, or detailed title rules and no user-company selection.
(4) Every original column is retained unchanged.
(5) Assume `startdate` uses YYYY-MM-DD; report missing or failed date casts before cohort filtering.

Wang Wenzhi
Time: 2026-09-08
"""

from pathlib import Path
from zipfile import ZIP_STORED, ZipFile

from pyspark import StorageLevel
from pyspark.sql import Column, SparkSession
from pyspark.sql import functions as F


# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 1. Define the source, export paths, and text-cleaning helper
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


POSITION_TABLE = "user_positions"
OUTPUT_PARQUET = "Files/WenzhiW/A01_BaselineUSBiopharma/StageA_BroadestCandidateEmpSpells"
OUTPUT_WRITE_MODE = "overwrite"
REQUIRED_COLUMNS = [
    "position_id",
    "user_id",
    "rcid",
    "position_number",
    "country",
    "title_raw",
    "title_translated",
    "startdate",
    "onet_code",
]
GENERATED_COLUMNS = ["parsed_start_date"]
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


def clean_spark_text(column_name: str) -> Column:
    """
    Trim source text and recognize explicit missing tokens without changing the source column.
    """
    value = F.trim(F.col(column_name).cast("string"))
    missing = value.isNull() | F.lower(value).isin("", "empty", "null", "none", "nan", "na", "n/a")
    return F.when(missing, F.lit(None).cast("string")).otherwise(value)

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 2. Filter source spells while retaining every source variable
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


source_spells = spark.read.table(POSITION_TABLE)
source_columns = source_spells.columns
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(source_columns))
collisions = sorted(set(GENERATED_COLUMNS) & set(source_columns))
if missing_columns or collisions or len(source_columns) != len(set(source_columns)):
    raise ValueError(f"Missing fields: {missing_columns}; derived-name collisions: {collisions}.")

"""
Notes:
(1) `filtered_spells` keeps rows with
    (a) non-missing country and job title information.
    (b) non-missing values on the following four columns: "position_id", "user_id", "rcid", and
        "position_number".
(2) `filtered_spells` keeps rows whose ONET occupation codes fall into 2-digit codes: 17 or 19.
(3) `candidate_spells` keeps rows whose starte date falls in range [2021-01-01, 2024-01-01).
"""
is_usable = clean_spark_text("country").isNotNull() & (
    clean_spark_text("title_raw").isNotNull() | clean_spark_text("title_translated").isNotNull()
)
for column_name in ["position_id", "user_id", "rcid", "position_number"]:
    is_usable &= clean_spark_text(column_name).isNotNull()
is_focal_onet = F.substring(clean_spark_text("onet_code"), 1, 2).isin("17", "19")
filtered_spells = source_spells.filter(is_usable & is_focal_onet)

filtered_spells = filtered_spells.withColumn(
    "parsed_start_date", F.expr("try_cast(startdate as date)")
)
missing_start_date_count = filtered_spells.filter(F.col("startdate").isNull()).count()
unparseable_start_date_count = filtered_spells.filter(
    F.col("startdate").isNotNull() & F.col("parsed_start_date").isNull()
).count()
print(f"Rows in filtered_spells with missing start dates: {missing_start_date_count:,}")
print(
    "Rows in filtered_spells with non-missing but unparseable start dates: "
    f"{unparseable_start_date_count:,}"
)

candidate_spells = filtered_spells.filter(
    (F.col("parsed_start_date") >= F.lit("2021-01-01").cast("date"))
    & (F.col("parsed_start_date") < F.lit("2024-01-01").cast("date"))
).select(*source_columns, *GENERATED_COLUMNS)

if candidate_spells.columns != source_columns + GENERATED_COLUMNS:
    raise AssertionError("All source variables must survive the extraction.")
candidate_spells.explain("formatted")

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 3. Materialize the filtered sample, report counts, and write Parquet
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>

candidate_spells = candidate_spells.persist(StorageLevel.DISK_ONLY)
try:
    spell_count = candidate_spells.count()
    print(f"Employment-spell count: {spell_count:,}")
    candidate_spells.write.mode(OUTPUT_WRITE_MODE).parquet(OUTPUT_PARQUET)
    written = spark.read.parquet(OUTPUT_PARQUET)
    if written.columns != candidate_spells.columns or written.count() != spell_count:
        raise AssertionError("Written Stage A schema or row count differs from the extract.")
    print(f"Saved {spell_count:,} employment spells with {len(source_columns)} source columns.")
finally:
    candidate_spells.unpersist()

In [ ]:

# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 4. Package the distributed Parquet output for download
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


source_directory = Path("/lakehouse/default") / OUTPUT_PARQUET
archive_path = source_directory.with_suffix(".zip")
staged_archive = archive_path.with_suffix(".zip.incomplete")
if archive_path.exists() and OUTPUT_WRITE_MODE != "overwrite":
    raise FileExistsError(archive_path)
source_files = sorted(path for path in source_directory.rglob("*.parquet") if path.is_file())
if not source_files:
    raise FileNotFoundError(f"No Parquet parts in {source_directory}.")
with ZipFile(staged_archive, mode="w", compression=ZIP_STORED, allowZip64=True) as archive:
    for source_file in source_files:
        archive.write(
            source_file, arcname=source_file.relative_to(source_directory.parent).as_posix()
        )
with ZipFile(staged_archive) as archive:
    if archive.testzip() is not None:
        raise OSError("ZIP validation failed.")
staged_archive.replace(archive_path)
print(f"Download and extract {archive_path}; keep the directory name and every Parquet part.")